# Bài 1: Titanic Dataset

### Yêu cầu

1. Sử dụng **Logistic Regression** để dự đoán hành khách sống sót hay không.
2. Sử dụng cùng cách chia dữ liệu train/test như bài tập trước nếu có thể.
3. Đánh giá kết quả của mô hình Logistic Regression.
4. So sánh kết quả của **Logistic Regression** với kết quả của **Linear Regression** trong bài tập trước.

---

#### 0. Chuẩn bị môi trường

In [35]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.neighbors import KNeighborsClassifier
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)          # cố định ngẫu nhiên -> kết quả tái lập được
print("Sẵn sàng.")

Sẵn sàng.


#### 1. Tải dữ liệu (đã cho)

In [36]:
try:
    df = sns.load_dataset("titanic")
    print("Đã tải từ seaborn.")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    df.columns = [c.lower() for c in df.columns]
    print("Đã tải từ URL.")
df.head()

Đã tải từ seaborn.


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


#### 2. Loại bỏ cột rò rỉ nhãn (data leakage) và cột dư thừa

In [37]:
missing_ratio = df.isnull().mean()

leaky = ['alive', 'adult_male', 'who', 'class', 'deck', 'embark_town', 'alone']      # điền danh sách cột cần bỏ (chỉ những cột có trong df)

df = df.drop(columns=leaky, errors='ignore')

# print("Các cột còn lại:", list(df.columns))

#### 3. Chia train/val/test có stratify

In [38]:
X = df.drop(columns="survived")
y = df["survived"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"  tỷ lệ Survived ({name}): {yy.mean():.3f}")

Train: (623, 7) | Val: (134, 7) | Test: (134, 7)
  tỷ lệ Survived (train): 0.384
  tỷ lệ Survived (val): 0.388
  tỷ lệ Survived (test): 0.381


#### 4. Xây pipeline cho biến số và biến phân loại

In [39]:
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["pclass"]

pipe_so  = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  RobustScaler()),
])
pipe_cat = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess = ColumnTransformer([
    ("num", pipe_so,  num_cols),
    ("cat", pipe_cat, cat_cols),
    ("ord", "passthrough", ord_cols),
])

preprocess.fit(X_train)               # fit CHỈ trên train
X_train_t = preprocess.transform(X_train)
X_val_t   = preprocess.transform(X_val)
X_test_t  = preprocess.transform(X_test)
print(X_train_t.shape, list(preprocess.get_feature_names_out()))

(623, 10) ['num__age', 'num__sibsp', 'num__parch', 'num__fare', 'cat__sex_female', 'cat__sex_male', 'cat__embarked_C', 'cat__embarked_Q', 'cat__embarked_S', 'ord__pclass']


#### 5. Huấn luyện mô hình Linear Regression

In [40]:
linear_model = LinearRegression()
linear_model.fit(X_train_t, y_train)
y_pred = linear_model.predict(X_test_t)
y_pred_binary = (y_pred >= 0.5).astype(int)
# print(y_pred)

In [41]:
print("Accuracy:", accuracy_score(y_test, y_pred_binary))
print("\nClassification Report:\n", classification_report(y_test, y_pred_binary))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_binary))

Accuracy: 0.7686567164179104

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.84      0.82        83
           1       0.72      0.65      0.68        51

    accuracy                           0.77       134
   macro avg       0.76      0.75      0.75       134
weighted avg       0.77      0.77      0.77       134

Confusion Matrix:
 [[70 13]
 [18 33]]


#### 6. Huấn luyện mô hình Logistic Regression

In [42]:
logis_model = LogisticRegression()
logis_model.fit(X_train_t, y_train)
y_pred = logis_model.predict(X_test_t)

In [43]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.7388059701492538

Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.78      0.79        83
           1       0.65      0.67      0.66        51

    accuracy                           0.74       134
   macro avg       0.72      0.72      0.72       134
weighted avg       0.74      0.74      0.74       134


Confusion Matrix:
 [[65 18]
 [17 34]]


### So sánh 
Khi in giá trị `y_pred` trong mô hình Linear Regression, ta thấy có giá trị âm (vô lý khi xem đó là xác suất). Mặc dù mô hình Linear Regression đạt được Accuracy cao hơn so với Logistic Regression, nhưng nó không phải là lựa chọn tối ưu. Về mặt lý thuyết, Linear Regression dự đoán các giá trị liên tục ngoài phạm vi [0, 1], điều này đi ngược lại với bản chất của bài toán xác suất nhị phân. Trong khi đó, Logistic Regression thông qua hàm Sigmoid đã ánh xạ dữ liệu vào không gian xác suất, cho thấy sự ổn định và khả năng nhận diện các trường hợp sống sót (Recall) tốt hơn. Do đó, Logistic Regression được chọn làm mô hình phù hợp hơn cho bài toán phân loại hành khách Titanic này.

# Bài 2: Dry Bean Dataset

Xây dựng mô hình phân loại các loại hạt trong bộ dữ liệu **Dry Bean Dataset** bằng hai thuật toán:

1. **Logistic Regression**
2. **K-Nearest Neighbors – KNN**
---

#### 0. Chuẩn bị môi trường và các tập dữ liệu

In [44]:
DATA_CANDIDATES = [
    Path("Dry_Bean_Dataset/dry_bean_train.csv"),
    Path("../Dry_Bean_Dataset/dry_bean_train.csv"),
    Path("week04/Homework_b7/Dry_Bean_Dataset/dry_bean_train.csv") 
]

TRAIN_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)

if TRAIN_PATH is None:
    raise FileNotFoundError("Không tìm thấy file dữ liệu! Hãy kiểm tra lại thư mục.")
else:
    print("Đã tìm thấy dữ liệu tại:", TRAIN_PATH.resolve())
    train_df = pd.read_csv(TRAIN_PATH)

Đã tìm thấy dữ liệu tại: C:\Users\HP\Desktop\mliot-pyml-2026-hw\week04\Homework_b7\Dry_Bean_Dataset\dry_bean_train.csv


In [45]:
target = "class"
X_train = train_df.drop(columns=[target])
y_train = train_df[target]
test_df = pd.read_csv(str(TRAIN_PATH).replace("train", "test"))
X_test = test_df.drop(columns=[target])
y_test = test_df[target]

#### 1. Xây dựng pipeline 

In [46]:
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# KNN với K = 5
pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier(n_neighbors=5))
])

#### 2. Huấn luyện và đánh giá 

In [47]:
pipe_lr.fit(X_train, y_train)
pipe_knn.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](7,)","['BARBUNYA','BOMBAY','CALI',...,'HOROZ','SEKER','SIRA']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](16,)","['area','perimeter','majoraxislength',...,'shapefactor2','shapefactor3', 'shapefactor4']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,16
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [48]:
y_pred_lr = pipe_lr.predict(X_test)
y_pred_knn = pipe_knn.predict(X_test)

print("--- Logistic Regression ---")
print(classification_report(y_test, y_pred_lr))

print("--- KNN ---")
print(classification_report(y_test, y_pred_knn))

--- Logistic Regression ---
              precision    recall  f1-score   support

    BARBUNYA       0.93      0.89      0.91       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.91      0.94      0.93       326
    DERMASON       0.93      0.91      0.92       709
       HOROZ       0.96      0.94      0.95       372
       SEKER       0.93      0.94      0.94       406
        SIRA       0.86      0.88      0.87       527

    accuracy                           0.92      2709
   macro avg       0.93      0.93      0.93      2709
weighted avg       0.92      0.92      0.92      2709

--- KNN ---
              precision    recall  f1-score   support

    BARBUNYA       0.93      0.88      0.90       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.90      0.95      0.92       326
    DERMASON       0.92      0.91      0.91       709
       HOROZ       0.96      0.93      0.95       372
       SEKER       0.95      0.94     

- Cả 2 mô hình đều đạt `acurracy` = 92% tức là mô hình dự đoán đúng loại hạt trong 92% trường hợp trên tập test.
- Cả Logistic Regression và KNN (K=3) đều cho kết quả rất tương đồng nhau. Sự khác biệt nhỏ nằm ở các chỉ số precision hoặc recall của từng loại hạt cụ thể (ví dụ: hạt SEKER có precision ở KNN là 0.95, cao hơn một chút so với 0.93 của Logistic Regression).

### Thử các giá trị K

In [49]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_neighbors': [3, 5, 7, 9, 11, 13, 15]
}

grid_search = GridSearchCV(pipe_knn, param_grid, cv=5, scoring='accuracy', n_jobs=-1)

grid_search.fit(X_train, y_train)

print(f"Giá trị K tốt nhất: {grid_search.best_params_}")
print(f"Accuracy tốt nhất: {grid_search.best_score_:.4f}")

Giá trị K tốt nhất: {'model__n_neighbors': 13}
Accuracy tốt nhất: 0.9259
